# MIE353 Assignment 3 Notebook

This notebook runs every part of your assignment:

| Step | What it does | Your input |
|---|---|---|
| **Question 1: Datalog** | | |
| 2 | shows the given knowledge base (`part1_datalog/kb_facts.lp`) | — |
| 3 | checks your **Datalog rules** with clingo | `part1_datalog/kb_rules.lp` |
| 4 | runs your **Datalog queries** | `part1_datalog/queries.lp` |
| **Question 2: RDF and SPARQL** | | |
| 5 | loads and checks the CMS csv files (`data/`) | — |
| 6 | runs your **YARRRML mappings** and loads your **ontology** into a knowledge graph | `part2_rdf/cms_mappings.yaml`, `part2_rdf/cms_ontology.owl` |
| 7 | runs your **SPARQL queries** on the knowledge graph | `part2_rdf/sparql/query1.sparql` … `query7.sparql` |
| 8 | writes your mapped data as **Datalog facts** (`results/kg_triples.lp`) | — |
| 9 | runs your **Datalog versions** of the SPARQL queries | `part2_rdf/datalog/queries.lp` |
| 10 | packages your submission zip for Quercus | — |

You write your rules, mappings, and queries in **any text editor or IDE** (e.g. VS Code) and save them as files; the notebook reads them from the paths in step 1. The only thing you edit inside the notebook is the configuration (step 1). Cells marked **✏️ EDIT** need your input; cells marked **▶️ RUN** don't.

Run the cells from top to bottom (*Run All* works). A step whose files don't exist yet says so and is skipped, so you can work on Question 1 before you start Question 2. After you change one of your files, re-run the step that uses it.

## 0. Setup ▶️ RUN

The first cell checks where the notebook is running. **In Google Colab**, it installs the packages the notebook needs (this takes a minute). **Anywhere else** (e.g. VS Code with the devcontainer of the starter repository), it only checks that they are installed; if one is missing, install them with `pip install -r requirements.txt` (the devcontainer does this for you).

In [ ]:
import importlib.util
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Keep this list in sync with requirements.txt
REQUIREMENTS = {"clingo": "clingo", "morph-kgc": "morph_kgc", "owlrl": "owlrl",
                "pandas": "pandas", "rdflib": "rdflib"}   # pip name -> module name

if IN_COLAB:
    print("Running in Google Colab: installing " + ", ".join(REQUIREMENTS) + " ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *REQUIREMENTS], check=True)
    print("Done.")
else:
    missing = [pkg for pkg, module in REQUIREMENTS.items() if importlib.util.find_spec(module) is None]
    if missing:
        raise ModuleNotFoundError(
            "Missing packages: " + ", ".join(missing) + ". Install them with\n"
            "    pip install -r requirements.txt\n"
            "in the environment of this notebook's kernel (the devcontainer does this for you), "
            "then restart the kernel."
        )
    print("Not running in Colab: all required packages are installed.")

In [ ]:
import logging
import os
import re
import zipfile
from collections import Counter
from contextlib import contextmanager
from pathlib import Path

import clingo
import morph_kgc
import owlrl
import pandas as pd
import rdflib
from rdflib.namespace import OWL, RDF, RDFS
from IPython.display import display

CSV_FILES = [
    "universities.csv", "students.csv", "professors.csv",
    "courses.csv", "offerings.csv", "transcript.csv",
]
STARTER_MAPPING_FILE = Path("starter_yarrrml.yaml")   # provided in the starter files
RESULTS_DIR = Path("results")
DATA_DIR = Path("data")
TRIPLES_FILE = RESULTS_DIR / "kg_triples.lp"           # generated in step 8

print("clingo", clingo.__version__, "| rdflib", rdflib.__version__, "| Colab:", IN_COLAB)

## 1. Configuration ✏️ EDIT

Set your UTORid. The other settings tell the notebook where your files are; the defaults match the folder structure of the starter files and of the submission zip, so you only need to change them if you keep your files somewhere else.

**Using Colab?** Files you upload to a Colab session are deleted when the session ends. To keep your work, put your assignment folder (the starter files) in Google Drive and uncomment the two lines at the top of the cell (change the folder name to yours).

In [ ]:
# Colab only: work in a folder in your Google Drive
# from google.colab import drive; drive.mount("/content/drive")
# %cd /content/drive/MyDrive/MIE353_A3

UTORID = "your_utorid"                              # e.g. "smithj12"

# Question 1: Datalog
KB_FACTS_FILE = "part1_datalog/kb_facts.lp"         # given; do not change this file
KB_RULES_FILE = "part1_datalog/kb_rules.lp"         # your rules
KB_QUERY_FILE = "part1_datalog/queries.lp"          # your queries: query1 ... query5

# Question 2: RDF and SPARQL
CSV_SOURCE = "data"                                 # folder (or URL) with the csv files
ONTOLOGY_FILE = "part2_rdf/cms_ontology.owl"        # saved from Protégé (RDF/XML or Turtle)
MAPPING_FILE = "part2_rdf/cms_mappings.yaml"        # your YARRRML mappings
SPARQL_DIR = "part2_rdf/sparql"                     # query1.sparql ... query7.sparql
RDF_DATALOG_FILE = "part2_rdf/datalog/queries.lp"   # your Datalog queries: query1 ... query7

REPORT_FILE = "A3_report.pdf"
NOTEBOOK_FILE = "A3_notebook_starter.ipynb"                 # this notebook (save it before packaging)

RDFS_REASONING = False   # True: apply RDFS inference (subclass, domain, range) before running SPARQL

### Create starter files ▶️ RUN

This creates starter files, **only for files that don't exist yet**, so it never overwrites your work:

- Question 1: a template for your rules (`kb_rules.lp`) and for your queries (`queries.lp`);
- Question 2: `cms_mappings.yaml` (a copy of the starter mapping `starter_yarrrml.yaml`), a template for each of your 7 SPARQL queries (`sparql/query1.sparql` …), and a template for your Datalog queries (`datalog/queries.lp`).

Open them in your editor and fill them in. Your ontology (`cms_ontology.owl`) is created in Protégé, not here.

In [ ]:
KB_RULES_TEMPLATE = '''% Question 1 rules. The facts are in kb_facts.lp; the comments in that file explain them.

% ---------------------------------------------------------------
% Required predicates
% ---------------------------------------------------------------

% passed(S, C): student S has completed course C with a grade of 50 or above.


% failed(S, C): student S has completed course C with a grade below 50.


% requires(C, P): P is a prerequisite of C, directly or indirectly
% (a prerequisite of a prerequisite, and so on).


% missingPrereq(S, C): student S cannot take course C because they have not passed
% (or have not taken) one of its prerequisites.


% canTake(S, C): student S can take course C, because they have not passed it
% previously, and they have passed every prerequisite of C.


% ---------------------------------------------------------------
% Integrity constraints (at least one; the data must satisfy it)
% A constraint  :- body.  rejects every answer set in which the body is true.
% ---------------------------------------------------------------


% ---------------------------------------------------------------
% Helper predicates (optional)
% ---------------------------------------------------------------
'''

PART1_QUESTIONS = [
    ("What are the codes and titles of all courses that mie451 requires, directly or indirectly?", "Code, Title"),
    ("Which students (full names) can take mie368?", "First, Last"),
    ("Which students (full names) have failed at least one course?", "First, Last"),
    ("Which instructors (full names) teach only first-year courses?", "First, Last"),
    ("Which students (full names) are enrolled in a course without having passed all of its prerequisites, and in which course?", "First, Last, Course"),
]

PART2_QUESTIONS = [
    ("What was the highest grade received in any offering of \"mie444\" at the University of Toronto?", "Grade"),
    ("What are all the course titles for second-year courses? (Course code \"XXX2YY\")", "Title"),
    ("What are the full names (first and last) of professors teaching at least two courses?", "First, Last"),
    ("What are the full names (first and last) of all students who have taken at least one course taught by \"Patrick Flores\"?", "First, Last"),
    ("Which students (full names) have never taken a course taught by \"Kelsey Lewis\"?", "First, Last"),
    ("List all students (full names) who have taken the \"Control Systems I\" course and also failed at least one other course (a failing grade is 49 or below).", "First, Last"),
    ("What are the full names and emails of all persons?", "First, Last, Email"),
]

PART1_QUERIES_HEADER = '''% Question 1 queries. Run with: clingo kb_facts.lp kb_rules.lp queries.lp
%
% Define one predicate per question: query1, query2, ... Its instances are the answers.
% You can use the predicates of kb_facts.lp, your rules in kb_rules.lp, and helper
% predicates that you define in this file.
'''

RDF_DATALOG_HEADER = '''% Question 2: your SPARQL queries rewritten in Datalog. Run with: clingo kg_triples.lp queries.lp
%
% kg_triples.lp (written by step 8 of the notebook) contains your mapped data as facts:
% one unary predicate per class and one binary predicate per property, named after the
% class or property with a lowercase first letter, e.g.
%   student(9000).
%   hasFirstName(9000, "Lauren").
% Step 8 lists all the predicates and shows an example of each.
%
% Define one predicate per question: query1, query2, ... Its instances are the answers.
% You may define helper predicates in this file. If a query cannot be written in plain
% Datalog, leave it out and explain why in your report.
'''

QUERY_STUB = '''
% Query {n}: {question}
% query{n}({args}) :- ...
'''

SPARQL_QUERY_TEMPLATE = '''# Question 2, competency question {n}:
# {question}

PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
PREFIX on:   <http://stl.mie.utoronto.ca/mie353#>   # ontology namespace used in starter_yarrrml.yaml
PREFIX ex:   <http://example.com#>                  # instance namespace used in starter_yarrrml.yaml
# Change these if your ontology or mappings use different namespaces.

# SELECT ...
# WHERE {{
# }}
'''


def query_program(header, questions):
    return header + "".join(
        QUERY_STUB.format(n=n, question=q, args=args) for n, (q, args) in enumerate(questions, start=1))


def create_if_missing(path, content):
    path = Path(path)
    if path.exists():
        return False
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)
    return True


created = []
if create_if_missing(KB_RULES_FILE, KB_RULES_TEMPLATE):
    created.append(KB_RULES_FILE)
if create_if_missing(KB_QUERY_FILE, query_program(PART1_QUERIES_HEADER, PART1_QUESTIONS)):
    created.append(KB_QUERY_FILE)
if not Path(MAPPING_FILE).exists() and STARTER_MAPPING_FILE.exists():
    if create_if_missing(MAPPING_FILE, STARTER_MAPPING_FILE.read_text()):
        created.append(f"{MAPPING_FILE} (copy of {STARTER_MAPPING_FILE})")
for n, (question, args) in enumerate(PART2_QUESTIONS, start=1):
    p = Path(SPARQL_DIR) / f"query{n}.sparql"
    if create_if_missing(p, SPARQL_QUERY_TEMPLATE.format(n=n, question=question)):
        created.append(str(p))
if create_if_missing(RDF_DATALOG_FILE, query_program(RDF_DATALOG_HEADER, PART2_QUESTIONS)):
    created.append(RDF_DATALOG_FILE)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Created:" if created else "All starter files already exist; nothing was changed.")
for p in created:
    print("  " + p)

# Question 1: Datalog

## 2. The knowledge base ▶️ RUN

Question 1 uses a small knowledge base about a degree program: students, instructors, courses, the prerequisites of each course, the courses each student has completed (with a grade), and the courses each student is enrolled in now. It is given to you as Datalog facts in `part1_datalog/kb_facts.lp`; the comments in that file explain every predicate. **Don't change this file.**

This cell shows how many facts there are for each predicate.

In [ ]:
def run_clingo(files, max_models=2):
    # Run clingo on the given files. Returns a dict with the models and any messages.
    messages = []

    def logger(code, message):
        messages.append(message.strip())

    ctl = clingo.Control([str(max_models)], logger=logger, message_limit=50)
    try:
        for f in files:
            ctl.load(str(f))
        ctl.ground([("base", [])])
    except RuntimeError as e:
        return {"error": str(e), "messages": messages, "models": [], "satisfiable": None}

    models = []
    result = ctl.solve(on_model=lambda m: models.append(
        (list(m.symbols(atoms=True)), list(m.symbols(shown=True)))))
    return {"error": None, "messages": messages, "models": models, "satisfiable": result.satisfiable}


def show_messages(res):
    if res["messages"]:
        print("clingo messages:")
        for m in res["messages"]:
            print("  " + m.replace("\n", "\n  "))


def atom_counts(atoms):
    return Counter(f"{s.name}/{len(s.arguments)}" for s in atoms)


def show_knowledge_base():
    if not Path(KB_FACTS_FILE).exists():
        print(f"Not found: {Path(KB_FACTS_FILE).resolve()}. It is in the starter files; check KB_FACTS_FILE in step 1.")
        return
    res = run_clingo([KB_FACTS_FILE])
    show_messages(res)
    if res["error"]:
        print(f"ERROR: {res['error']}")
        return
    atoms, _ = res["models"][0]
    print(f"{KB_FACTS_FILE}: {len(atoms)} facts\n")
    for p, c in sorted(atom_counts(atoms).items()):
        print(f"  {p:20} {c:4}")


show_knowledge_base()

## 3. Check your rules ▶️ RUN

Write your rules in your editor and save them in `part1_datalog/kb_rules.lp` (`KB_RULES_FILE`). The template lists the predicates you must define.

This cell runs `clingo kb_facts.lp kb_rules.lp`. It shows clingo's error messages if there are any (for example, an unsafe variable), tells you if an integrity constraint is violated, and otherwise shows how many atoms there are for each predicate, including the ones your rules derive.

All rules must be **safe**: every variable in a rule must appear in at least one positive (non-negated) literal in the body of the rule.

In [ ]:
REQUIRED_PREDICATES = ["passed/2", "failed/2", "requires/2", "missingPrereq/2", "canTake/2"]


def check_rules():
    if not Path(KB_RULES_FILE).exists():
        print(f"Rules file not found: {Path(KB_RULES_FILE).resolve()}")
        return
    res = run_clingo([KB_FACTS_FILE, KB_RULES_FILE])
    show_messages(res)
    if res["error"]:
        print(f"\nERROR: {res['error']}. Fix the problem reported above.")
        return
    if not res["satisfiable"]:
        print("\nNo answer set: one of your integrity constraints is violated by the data.")
        print("Your constraints must be satisfied by the data, so check the constraint.")
        return
    atoms, _ = res["models"][0]
    counts = atom_counts(atoms)
    print(f"OK: the knowledge base has an answer set with {len(atoms)} atoms.\n")
    print("Atoms per predicate (facts + derived):")
    for p, c in sorted(counts.items()):
        print(f"  {p:20} {c:5}")
    missing = [p for p in REQUIRED_PREDICATES if p not in counts]
    if missing:
        print("\nNot derived yet (required): " + ", ".join(missing))
    if len(res["models"]) > 1:
        print("\nNote: your program has more than one answer set. Plain Datalog programs have exactly one, "
              "so check whether you meant to use a choice rule or unstratified negation.")


check_rules()

## 4. Run your queries ▶️ RUN

Write all your queries in one file, `part1_datalog/queries.lp`, with one predicate per question: `query1`, `query2`, …, `query5`. The instances of `queryN` are the answers to question N, e.g.

```prolog
query1(Code, Title) :- ...
```

This cell runs `clingo kb_facts.lp kb_rules.lp queries.lp`, shows the answers to each query, and saves them to `results/part1_queryN.csv` for your report. (You don't need a `#show` directive: the notebook reads the `queryN` atoms directly.)

In [ ]:
def symbol_value(sym):
    if sym.type == clingo.SymbolType.Number:
        return sym.number
    if sym.type == clingo.SymbolType.String:
        return sym.string
    return str(sym)


def run_query_program(label, files, qfile, n_queries, out_prefix):
    # Run one query program together with `files`, and show the instances of query1 ... queryN.
    # Returns {n: DataFrame} for the queries that have answers.
    results = {}
    qfile = Path(qfile)
    missing = [str(f) for f in list(files) + [qfile] if not Path(f).exists()]
    if missing:
        print("Not found: " + ", ".join(missing))
        return results
    res = run_clingo(list(files) + [qfile])
    if res["error"]:
        show_messages(res)
        print(f"ERROR: {res['error']}")
        return results
    if not res["satisfiable"]:
        print("No answer set: an integrity constraint is violated.")
        return results

    atoms, _ = res["models"][0]
    text = qfile.read_text()
    for n in range(1, n_queries + 1):
        name = f"query{n}"
        print(f"===== {label} {n} =====")
        found = [s for s in atoms if s.name == name]
        if not found:
            defined = re.search(rf"^\s*{name}\s*\(", text, re.MULTILINE)
            print("No answers.\n" if defined else f"No {name} rule written yet.\n")
            continue
        rows = sorted({tuple(symbol_value(a) for a in s.arguments) for s in found}, key=str)
        n_args = max(len(r) for r in rows)
        df = pd.DataFrame(rows, columns=[f"arg{k}" for k in range(1, n_args + 1)])
        print(f"{len(df)} answer(s):")
        display(df)
        df.to_csv(RESULTS_DIR / f"{out_prefix}{n}.csv", index=False)
        print()
        results[n] = df
    return results


part1_results = run_query_program("Question 1, query", [KB_FACTS_FILE, KB_RULES_FILE], KB_QUERY_FILE,
                                  len(PART1_QUESTIONS), "part1_query")

# Question 2: RDF and SPARQL

## 5. Load the csv data ▶️ RUN

This loads the six csv files of the Course Management System and shows the columns and first rows of each one. You will need the column names for your mappings.

It also runs a few checks on the data. Read them before you design your identifiers:

- **`courseId` alone does not identify a course.** Some course codes (e.g. `csc100`, `mat200`, `phy100`) are used by several universities for different courses, so a course is identified by `courseId` **together with** `universityId`. `offerings.csv` refers to a course by both columns.
- `universities.csv` gives the name of each university (`universityId`), which you need for question 1 ("at the University of Toronto").
- An exact duplicate row is harmless: duplicate triples are removed automatically.

In [ ]:
def _is_url(source):
    return source.startswith(("http://", "https://"))


# Folder that holds the csv files; the mappings are run from this folder (step 6)
CSV_DIR = DATA_DIR if _is_url(CSV_SOURCE) else Path(CSV_SOURCE)


def read_csv(path_or_url):
    # Read every value as a string ("0" stays "0", empty stays ""); utf-8-sig drops the byte-order
    # mark at the start of students.csv, so its first column is "studentId" and not "﻿studentId".
    df = pd.read_csv(path_or_url, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]
    return df.apply(lambda col: col.str.strip())


def load_csvs(source):
    if _is_url(source):
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        tables = {}
        for name in CSV_FILES:
            df = read_csv(f"{source.rstrip('/')}/{name}")
            df.to_csv(DATA_DIR / name, index=False)  # local copy, used by the mappings
            tables[name] = df
        return tables

    CSV_DIR.mkdir(parents=True, exist_ok=True)
    missing = [n for n in CSV_FILES if not (CSV_DIR / n).exists()]
    if missing and IN_COLAB:
        from google.colab import files
        print("Please upload these files:", ", ".join(missing))
        for fname, content in files.upload().items():
            (CSV_DIR / fname).write_bytes(content)
        missing = [n for n in CSV_FILES if not (CSV_DIR / n).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing csv files in '{CSV_DIR.resolve()}': {', '.join(missing)}. "
            "Put the starter csv files from Quercus in that folder, or change CSV_SOURCE."
        )
    return {n: read_csv(CSV_DIR / n) for n in CSV_FILES}


def check_data(t):
    print("Data checks:")
    for name, df in t.items():
        dups = int(df.duplicated().sum())
        if dups:
            print(f"  {name}: {dups} exact duplicate row(s)")
    courses = t["courses.csv"]
    shared = courses.drop_duplicates(["courseId", "universityId"])
    shared = shared[shared.duplicated("courseId", keep=False)]
    if len(shared):
        codes = sorted(shared["courseId"].unique())
        print(f"  courses.csv: {len(codes)} courseId value(s) are used by more than one university: "
              f"{', '.join(codes)}.\n"
              "    -> identify a course by courseId AND universityId, e.g. ex:$(universityId)_$(courseId) "
              "in your mappings.")
    refs = [
        ("offerings.csv", ["courseId", "universityId"], "courses.csv", ["courseId", "universityId"]),
        ("offerings.csv", ["professorId"], "professors.csv", ["profId"]),
        ("transcript.csv", ["studentId"], "students.csv", ["studentId"]),
        ("transcript.csv", ["offeringId"], "offerings.csv", ["offeringId"]),
        ("courses.csv", ["universityId"], "universities.csv", ["universityId"]),
        ("professors.csv", ["universityId"], "universities.csv", ["universityId"]),
    ]
    for src_name, src_cols, dst_name, dst_cols in refs:
        keys = set(map(tuple, t[dst_name][dst_cols].values))
        bad = [k for k in map(tuple, t[src_name][src_cols].values) if k not in keys]
        if bad:
            print(f"  {src_name}: {len(bad)} row(s) refer to a {', '.join(src_cols)} not in {dst_name}")
    print("  done.")


tables = load_csvs(CSV_SOURCE)
for name, df in tables.items():
    print(f"\n=== {name}: {len(df)} rows ===")
    print("Columns:", list(df.columns))
    display(df.head(3))
print()
check_data(tables)

## 6. Build the knowledge graph ▶️ RUN

This runs your YARRRML mappings (`MAPPING_FILE`) on the csv files with [Morph-KGC](https://morph-kgc.readthedocs.io/), loads your ontology (`ONTOLOGY_FILE`), and combines them into your knowledge graph. The mapped triples are also saved to `results/cms_graph.ttl` so you can inspect them.

In your mappings, refer to the csv files by **file name only**, e.g. `students.csv`. The notebook runs the mappings from the folder that contains the csv files. Values from a csv file are strings unless you give them a datatype in your mapping (e.g. `xsd:integer` for grades), so give numbers a datatype if you want to compare them.

If your mapping file or your ontology doesn't exist yet, this step says so and is skipped (and so are the steps that need the knowledge graph).

The summary lists how many instances each class has and how many triples each property has. It also warns you about any class or property that your mappings use but your ontology does not declare: these usually mean a typo or a missing declaration.

In [ ]:
@contextmanager
def working_directory(path):
    old = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old)


def run_mappings():
    mapping = Path(MAPPING_FILE).resolve()
    config = f"[CONFIGURATION]\nlogging_level: WARNING\n\n[CMS]\nmappings: {mapping}\n"
    logging.getLogger().setLevel(logging.WARNING)
    try:
        with working_directory(CSV_DIR):
            return morph_kgc.materialize(config)
    except FileNotFoundError as e:
        raise FileNotFoundError(
            f"{e}\nHint: refer to the csv files by file name only (e.g. students.csv) in your mappings."
        ) from None


def load_ontology():
    path = Path(ONTOLOGY_FILE)
    for fmt in ("xml", "turtle"):
        g = rdflib.Graph()
        try:
            g.parse(path, format=fmt)
            return g
        except Exception:
            continue
    raise ValueError(
        f"Could not read {path}. Save your ontology from Protégé in RDF/XML or Turtle syntax "
        "(File > Save as...), not in functional syntax."
    )


def short(term, graph):
    if isinstance(term, rdflib.Literal):
        return term.toPython()
    if isinstance(term, rdflib.URIRef):
        return term.n3(graph.namespace_manager)
    return str(term)


def mapping_prefixes():
    # The prefixes declared in the YARRRML file (Morph-KGC does not keep them in its output).
    # Read line by line from the "prefixes:" block, so this also works when the rest of the
    # file is not strict YAML.
    prefixes, in_block = {}, False
    for line in Path(MAPPING_FILE).read_text().splitlines():
        if re.match(r"^prefixes\s*:", line):
            in_block = True
        elif in_block and re.match(r"^\S", line):
            break
        elif in_block:
            m = re.match(r"^\s+([A-Za-z][\w.-]*)\s*:\s*[\"']?([^\s\"']+)", line)
            if m:
                prefixes[m.group(1)] = m.group(2)
    return prefixes


def build_knowledge_graph():
    # Returns (kg, data): the knowledge graph used by SPARQL (ontology + mapped data, plus RDFS
    # inferences if RDFS_REASONING) and the mapped data alone, which step 8 exports for Datalog.
    data = run_mappings()
    onto = load_ontology()

    kg = rdflib.Graph()
    for prefix, ns in list(mapping_prefixes().items()) + list(onto.namespaces()) + list(data.namespaces()):
        kg.bind(prefix, ns, override=False)
    kg += onto
    kg += data
    data.serialize(destination=str(RESULTS_DIR / "cms_graph.ttl"), format="turtle")

    print(f"Ontology: {len(onto)} triples   Mapped data: {len(data)} triples")

    classes = Counter(o for o in data.objects(None, RDF.type))
    props = Counter(p for p in data.predicates() if p != RDF.type)
    # data.predicates() yields one entry per triple, so props counts triples per property
    print("\nInstances per class:")
    for c, n in sorted(classes.items(), key=lambda x: str(x[0])):
        print(f"  {short(c, kg):40} {n:6}")
    print("\nTriples per property:")
    for p, n in sorted(props.items(), key=lambda x: str(x[0])):
        print(f"  {short(p, kg):40} {n:6}")

    declared = set()
    for kind in (OWL.Class, RDFS.Class, OWL.ObjectProperty, OWL.DatatypeProperty, RDF.Property):
        declared |= set(onto.subjects(RDF.type, kind))
    undeclared = sorted({str(t) for t in list(classes) + list(props) if t not in declared})
    if undeclared:
        print("\nWARNING: your mappings use these classes/properties, but your ontology does not declare them:")
        for t in undeclared:
            print("  " + short(rdflib.URIRef(t), kg))
    else:
        print("\nAll classes and properties used by your mappings are declared in your ontology.")

    if RDFS_REASONING:
        before = len(kg)
        owlrl.DeductiveClosure(owlrl.RDFS_Semantics).expand(kg)
        print(f"\nRDFS reasoning added {len(kg) - before} inferred triples (used by SPARQL only).")
    return kg, data


missing = [f for f in (MAPPING_FILE, ONTOLOGY_FILE) if not Path(f).exists()]
if missing:
    kg = mapped_data = None
    print("Skipping this step: not found yet: " + ", ".join(str(Path(f).resolve()) for f in missing))
    print("Create the file(s) (or fix the paths in step 1) and re-run this step.")
else:
    kg, mapped_data = build_knowledge_graph()

## 7. Run your SPARQL queries ▶️ RUN

Write your SPARQL queries in `part2_rdf/sparql/query1.sparql` … `query7.sparql`, and declare every prefix you use at the top of each file.

This cell runs them on the knowledge graph from step 6, shows the answers, and saves them to `results/sparql_queryN.csv` for your report. By default, no reasoning is applied, so a query only sees the triples in your ontology and your mapped data (the same as the SPARQL tab in Protégé). Set `RDFS_REASONING = True` in step 1 and re-run steps 6 and 7 to also see triples inferred from your subclass, domain, and range axioms.

In [ ]:
sparql_results = {}
_SPARQL_QUERY_FORM = re.compile(r"\b(SELECT|ASK|CONSTRUCT|DESCRIBE)\b", re.IGNORECASE)


def run_sparql(n):
    qfile = Path(SPARQL_DIR) / f"query{n}.sparql"
    print(f"===== SPARQL query {n} =====")
    if kg is None:
        print("No knowledge graph: step 6 was skipped.\n")
        return
    if not qfile.exists():
        print(f"{qfile} does not exist.\n")
        return
    text = qfile.read_text()
    body = "\n".join(l for l in text.splitlines() if not l.lstrip().startswith("#"))
    if not _SPARQL_QUERY_FORM.search(body):
        print("No query written yet.\n")
        return
    try:
        res = kg.query(text)
    except Exception as e:
        print(f"ERROR: {type(e).__name__}: {e}\n")
        return
    if res.type == "ASK":
        print("Answer:", res.askAnswer, "\n")
        return
    if res.type != "SELECT":
        print(f"This is a {res.type} query; please write a SELECT query.\n")
        return
    names = [str(v) for v in res.vars]
    rows = [tuple(None if row[v] is None else short(row[v], kg) for v in res.vars) for row in res]
    df = pd.DataFrame(rows, columns=names)
    sparql_results[n] = df
    print(f"{len(df)} answer(s):")
    display(df)
    df.to_csv(RESULTS_DIR / f"sparql_query{n}.csv", index=False)
    print()


for n in range(1, len(PART2_QUESTIONS) + 1):
    run_sparql(n)

## 8. The mapped data as Datalog facts ▶️ RUN

This cell writes the RDF triples produced by your mappings (not your ontology, and **without** RDFS inferences) to `results/kg_triples.lp` as Datalog facts, using the vocabulary of your ontology:

- each `rdf:type` triple becomes a fact of a **unary** predicate named after the class, e.g. `ex:9000 rdf:type on:Student` becomes `student(9000).`
- each other triple becomes a fact of a **binary** predicate named after the property, e.g. `ex:9000 on:hasFirstName "Lauren"` becomes `hasFirstName(9000, "Lauren").`

Predicate names are the local names of your classes and properties with a lowercase first letter (clingo requires this). Individuals are written by their local name (`ex:9000` becomes `9000`, `ex:UT01_mie444` becomes `"UT01_mie444"`); literals are written as strings, except whole numbers with a numeric datatype (e.g. `xsd:integer`), which are written as numbers so you can compare them (`G <= 49`).

The output lists every predicate with an example fact; use it to write your Datalog queries in step 9. Re-run this step after you change your mappings.

In [ ]:
_INT = re.compile(r"-?(0|[1-9][0-9]*)")


def local_name(iri):
    # http://example.com#9000 -> 9000, http://stl.mie.utoronto.ca/mie353#Student -> Student
    for sep in ("#", "/"):
        if sep in iri and iri.rsplit(sep, 1)[1]:
            return iri.rsplit(sep, 1)[1]
    return iri


def predicate_name(iri):
    name = re.sub(r"[^A-Za-z0-9_]", "_", local_name(str(iri)))
    if not name[:1].isalpha():
        name = "p_" + name
    return name[0].lower() + name[1:]


def value_term(t):
    if isinstance(t, rdflib.Literal):
        v = t.toPython()
        if isinstance(v, int) and not isinstance(v, bool) and -2**31 < v < 2**31:
            return str(v)
        s = str(t)
    elif isinstance(t, rdflib.URIRef):
        s = local_name(str(t))
        if _INT.fullmatch(s) and -2**31 < int(s) < 2**31:
            return s
    else:
        s = "_:" + str(t)
    return '"' + s.replace("\\", "\\\\").replace('"', '\\"') + '"'


def export_facts():
    if mapped_data is None:
        TRIPLES_FILE.unlink(missing_ok=True)   # don't leave facts from an earlier graph behind
        print("No knowledge graph: step 6 was skipped.")
        return
    facts, sources = set(), {}
    for s, p, o in mapped_data:
        if p == RDF.type:
            if not isinstance(o, rdflib.URIRef):
                continue
            pred, iri, args = predicate_name(o), o, (value_term(s),)
        else:
            pred, iri, args = predicate_name(p), p, (value_term(s), value_term(o))
        sources.setdefault((pred, len(args)), set()).add(str(iri))
        facts.add(f"{pred}({', '.join(args)}).")
    facts = sorted(facts)
    header = "% Generated by the assignment notebook (step 8) from your mappings. Do not edit by hand.\n"
    TRIPLES_FILE.write_text(header + "\n".join(facts) + "\n")

    print(f"Wrote {len(facts)} facts to {TRIPLES_FILE}\n")
    counts = Counter((f.split("(")[0], f.count(",") + 1) for f in facts)
    examples = {}
    for f in facts:
        examples.setdefault(f.split("(")[0] + "/" + str(f.count(",") + 1), f)
    print("Predicates (with one example each):")
    for (pred, n_args), c in sorted(counts.items()):
        key = f"{pred}/{n_args}"
        print(f"  {key:25} {c:6}   e.g. {examples[key]}")
    clashes = {k: v for k, v in sources.items() if len(v) > 1}
    for (pred, n_args), iris in sorted(clashes.items()):
        print(f"\nWARNING: {pred}/{n_args} comes from more than one class or property: {', '.join(sorted(iris))}")


export_facts()

## 9. Run your Datalog versions of the SPARQL queries ▶️ RUN

Rewrite your SPARQL queries from step 7 as Datalog queries over the facts from step 8, and save them all in one file, `part2_rdf/datalog/queries.lp`, with one predicate per question: `query1`, `query2`, …, `query7`. You may define helper predicates in the same file. If a query cannot be written in plain Datalog, leave it out and explain why in your report.

The facts contain only your mapped data, never your ontology or the RDFS inferences: if a question needs reasoning over your class hierarchy, write that reasoning as rules.

This cell runs `clingo kg_triples.lp queries.lp`, shows the answers to each query, and saves them to `results/datalog_queryN.csv` for your report.

In [ ]:
rdf_datalog_results = run_query_program("Datalog query", [TRIPLES_FILE], RDF_DATALOG_FILE,
                                        len(PART2_QUESTIONS), "datalog_query")

## 10. Package your submission ▶️ RUN (when you're done)

**Save this notebook first** (Ctrl+S / Cmd+S), so the zip contains your latest version. In Colab, download it (*File → Download → Download .ipynb*) and upload it into the folder you're working in, then set `NOTEBOOK_FILE` in step 1 if its name or location is different.

This cell collects your files from the paths in step 1, puts them in the folder structure required by the assignment, and creates `A3_<UTORid>.zip`, which is what you upload to Quercus. It lists any required files that are missing.

In [ ]:
def submission_files():
    files = {
        "A3_notebook.ipynb": NOTEBOOK_FILE,
        "A3_report.pdf": REPORT_FILE,
        "part1_datalog/kb_rules.lp": KB_RULES_FILE,
        "part1_datalog/queries.lp": KB_QUERY_FILE,
        "part2_rdf/cms_ontology.owl": ONTOLOGY_FILE,
        "part2_rdf/cms_mappings.yaml": MAPPING_FILE,
        "part2_rdf/datalog/queries.lp": RDF_DATALOG_FILE,
    }
    for n in range(1, len(PART2_QUESTIONS) + 1):
        files[f"part2_rdf/sparql/query{n}.sparql"] = Path(SPARQL_DIR) / f"query{n}.sparql"
    return files


def package_submission():
    if not UTORID or UTORID == "your_utorid":
        print("Set UTORID in step 1 first.")
        return
    files = submission_files()
    missing = [f"{arc}  (expected at {path})" for arc, path in files.items() if not Path(path).exists()]
    zip_name = f"A3_{UTORID}.zip"
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        for arc, path in files.items():
            if Path(path).exists():
                zf.write(path, arcname=arc)

    with zipfile.ZipFile(zip_name) as zf:
        print(f"Created {zip_name} containing:")
        for name in sorted(zf.namelist()):
            print("  " + name)
    if missing:
        print("\nWARNING: these required files are missing, so they are NOT in the zip:")
        for m in missing:
            print("  " + m)
    else:
        print("\nAll required files are included.")
    if IN_COLAB:
        from google.colab import files as colab_files
        colab_files.download(zip_name)


package_submission()